# Deepfake Detection System

#### Group 9 | University of the West of England | UFCEM1-60-M

### Notebook Structure

| Section | Description | Cells |
|---:|---|---|
| 0 | Environment Check | Cell 0 |
| 1 | Setup and Configuration | Cells 1–2 |
| 2 | Dataset Exploration | Cells 3–4 |
| 3 | Preprocessing | Cells 5–9 |
| 4 | Model Definitions | Cells 10–12 |
| 5 | Spatial Branch Training | Cells 13–14 |
| 6 | Dual-Branch Training | Cell 15 |
| 7 | Evaluation and Ablation | Cells 16–18 |
| 8 | Explainability | Cells 19–23 |
| 9 | Summary | Cells 24–25 |

> **Resume checkpoints:** A, B, C, and D.

# SECTION 0 - ENVIRONMENT CHECK

In [87]:
# CELL 0 - What survived the session restart?
# Run this first in any new session. It tells you what work is already done.

import os
 
def check_state():
    paths = {
        "FF crops (train/real)"  : "/kaggle/working/processed/ff/train/real",
        "FF crops (train/fake)"  : "/kaggle/working/processed/ff/train/fake",
        "FF crops (val/real)"    : "/kaggle/working/processed/ff/val/real",
        "FF crops (val/fake)"    : "/kaggle/working/processed/ff/val/fake",
        "FF crops (test/real)"   : "/kaggle/working/processed/ff/test/real",
        "FF crops (test/fake)"   : "/kaggle/working/processed/ff/test/fake",
        "Celeb crops (real)"     : "/kaggle/working/processed/celeb/real",
        "Celeb crops (fake)"     : "/kaggle/working/processed/celeb/fake",
    }
    print("FACE CROP INVENTORY")
    total_crops = 0
    for name, p in paths.items():
        if os.path.exists(p):
            n = len([f for f in os.listdir(p) if f.endswith(".jpg")])
            total_crops += n
            print(f"  {name:<24}: {n:>8,}")
        else:
            print(f"  {name:<24}: {'MISSING':>8}")
    print(f"  {'TOTAL':<24}: {total_crops:>8,}")
 
    print("\nFILE INVENTORY")
    files = {
        "ff_manifest_v2.csv"     : "/kaggle/working/processed/ff_manifest_v2.csv",
        "celeb_manifest.csv"     : "/kaggle/working/processed/celeb_manifest.csv",
        "spatial_best.pth"       : "/kaggle/working/checkpoints/spatial_best.pth",
        "dual_best.pth"          : "/kaggle/working/checkpoints/dual_best.pth",
        "deploy.prototxt"        : "/kaggle/working/models/deploy.prototxt",
        "face_detector.caffemodel": "/kaggle/working/models/face_detector.caffemodel",
    }
    for name, p in files.items():
        if os.path.exists(p):
            size = os.path.getsize(p) / 1e6
            print(f"  {name:<26}: present ({size:.1f} MB)")
        else:
            print(f"  {name:<26}: missing")
 
    import shutil
    used = shutil.disk_usage("/kaggle/working").used
    print(f"\nDisk used: {used/1e9:.2f} GB of 20 GB")
 
    print("\nWHAT TO DO NEXT")
    if total_crops < 100000:
        print("  Crops missing or incomplete -> run CELL 1 through CELL 8 (full preprocessing)")
    elif not os.path.exists("/kaggle/working/processed/ff_manifest_v2.csv"):
        print("  Crops present, manifest missing -> run CELL 1, CELL 5, then CELL 9")
    elif not os.path.exists("/kaggle/working/checkpoints/spatial_best.pth"):
        print("  Crops and manifest ready -> run RESUME A, then CELL 10 onward")
    else:
        print("  Models trained -> run RESUME A, CELL 12, RESUME B, then CELL 16 onward")
 
check_state()

FACE CROP INVENTORY
  FF crops (train/real)   :   13,080
  FF crops (train/fake)   :   52,292
  FF crops (val/real)     :    2,722
  FF crops (val/fake)     :   10,954
  FF crops (test/real)    :    2,819
  FF crops (test/fake)    :   11,312
  Celeb crops (real)      :   15,671
  Celeb crops (fake)      :  103,378
  TOTAL                   :  212,228

FILE INVENTORY
  ff_manifest_v2.csv        : present (0.6 MB)
  celeb_manifest.csv        : present (0.9 MB)
  spatial_best.pth          : present (222.7 MB)
  dual_best.pth             : missing
  deploy.prototxt           : present (0.0 MB)
  face_detector.caffemodel  : present (10.7 MB)

Disk used: 4.78 GB of 20 GB

WHAT TO DO NEXT
  Models trained -> run RESUME A, CELL 12, RESUME B, then CELL 16 onward


# SECTION 1 - SETUP AND CONFIGURATION

In [89]:
# CELL 1 - Imports, paths, configuration

import os, cv2, json, random, time, io, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from tqdm.notebook import tqdm
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix, roc_curve)
import timm
import matplotlib.pyplot as plt
from PIL import Image as PILImage
warnings.filterwarnings('ignore')
 
# Paths
FF_ROOT         = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
CELEB_ROOT      = "/kaggle/input/datasets/reubensuju/celeb-df-v2"
OUT_ROOT        = "/kaggle/working/processed"
FF_PROCESSED    = "/kaggle/working/processed/ff"
CELEB_PROCESSED = "/kaggle/working/processed/celeb"
CKPT_DIR        = "/kaggle/working/checkpoints"
MODEL_DIR       = "/kaggle/working/models"
GRADCAM_DIR     = "/kaggle/working/gradcam"
MC_DIR          = "/kaggle/working/mc_dropout"
EXP_DIR         = "/kaggle/working/experiments"
 
MANIFEST_FF     = f"{OUT_ROOT}/ff_manifest_v2.csv"
MANIFEST_CELEB  = f"{OUT_ROOT}/celeb_manifest.csv"
 
for d in [OUT_ROOT, CKPT_DIR, MODEL_DIR, GRADCAM_DIR, MC_DIR, EXP_DIR]:
    os.makedirs(d, exist_ok=True)

# Config
CFG = {
    "frames_per_video" : 20,
    "face_size"        : 224,
    "face_conf_thresh" : 0.90,
    "batch_size"       : 32,
    "seed"             : 42,
}
FF_REAL_FOLDERS = ["original"]
FF_FAKE_FOLDERS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
 
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Configuration ready")
print(f"  Device : {device}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
 

Configuration ready
  Device : cpu


In [90]:
# CELL 2 - Download OpenCV DNN face detector weights
# Needs internet on. (Skips if files already present)

import urllib.request
 
PROTOTXT_URL = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
WEIGHTS_URL  = "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"
PROTOTXT     = os.path.join(MODEL_DIR, "deploy.prototxt")
WEIGHTS      = os.path.join(MODEL_DIR, "face_detector.caffemodel")
 
if not os.path.exists(PROTOTXT):
    urllib.request.urlretrieve(PROTOTXT_URL, PROTOTXT)
    print("Downloaded deploy.prototxt")
else:
    print("deploy.prototxt already present")
 
if not os.path.exists(WEIGHTS):
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
    print("Downloaded face_detector.caffemodel")
else:
    print("face_detector.caffemodel already present")
 
print(f"  prototxt : {os.path.getsize(PROTOTXT)/1024:.1f} KB")
print(f"  weights  : {os.path.getsize(WEIGHTS)/1024:.1f} KB")
 

deploy.prototxt already present
face_detector.caffemodel already present
  prototxt : 27.4 KB
  weights  : 10416.2 KB


# SECTION 2 - DATASET EXPLORATION

In [91]:
# CELL 3 - Dataset structure

print("FF++ STRUCTURE")
for item in sorted(os.listdir(FF_ROOT)):
    p = os.path.join(FF_ROOT, item)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.mp4')])
        print(f"  {item:<22} {n:>5} videos")
 
print("\nCELEB-DF v2 STRUCTURE")
celeb_total = 0
for item in sorted(os.listdir(CELEB_ROOT)):
    p = os.path.join(CELEB_ROOT, item)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.mp4')])
        celeb_total += n
        print(f"  {item:<22} {n:>5} videos")
    elif item.endswith('.txt'):
        print(f"  {item}")
print(f"  Total: {celeb_total} videos")

FF++ STRUCTURE
  DeepFakeDetection       1000 videos
  Deepfakes               1000 videos
  Face2Face               1000 videos
  FaceShifter             1000 videos
  FaceSwap                1000 videos
  NeuralTextures          1000 videos
  csv                        0 videos
  original                1000 videos

CELEB-DF v2 STRUCTURE
  Celeb-real               590 videos
  Celeb-synthesis         5639 videos
  List_of_testing_videos.txt
  YouTube-real             300 videos
  Total: 6529 videos


In [92]:
# CELL 4 - Sample video properties

folder = os.path.join(FF_ROOT, "original")
sample = os.path.join(folder, sorted(os.listdir(folder))[0])
cap = cv2.VideoCapture(sample)
print(f"Sample video : {os.path.basename(sample)}")
print(f"  Frames     : {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
print(f"  FPS        : {cap.get(cv2.CAP_PROP_FPS):.1f}")
print(f"  Resolution : {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
cap.release()

Sample video : 000.mp4
  Frames     : 396
  FPS        : 25.0
  Resolution : 640x480


# SECTION 3- PREPROCESSING


In [95]:
# CELL 5 - Frame extraction and face detection functions

# Load OpenCV DNN face detector
_face_net = cv2.dnn.readNetFromCaffe(PROTOTXT, WEIGHTS)
_face_net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
_face_net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
print("Face detector loaded (OpenCV ResNet-SSD, CPU backend)")

def extract_frames(video_path, n_frames=20):
    """Uniformly sample n_frames across the full video duration."""
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs  = (list(range(total)) if total < n_frames
             else np.linspace(0, total - 1, n_frames, dtype=int).tolist())
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        if ok:
            frames.append(frame)
    cap.release()
    return frames


def detect_and_crop_face(frame, size=224, conf_thresh=0.90):
    """Return the highest-confidence face crop as RGB uint8, or None."""
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)),
                                  1.0, (300, 300), (104.0, 177.0, 123.0))
    _face_net.setInput(blob)
    dets = _face_net.forward()

    best, best_conf = None, 0.0
    for i in range(dets.shape[2]):
        conf = float(dets[0, 0, i, 2])
        if conf < conf_thresh or conf <= best_conf:
            continue
        best_conf = conf
        best = (int(dets[0, 0, i, 3] * w), int(dets[0, 0, i, 4] * h),
                int(dets[0, 0, i, 5] * w), int(dets[0, 0, i, 6] * h))

    if best is None:
        return None
    x1, y1, x2, y2 = best
    px, py = int((x2 - x1) * 0.15), int((y2 - y1) * 0.15)
    x1, y1 = max(0, x1 - px), max(0, y1 - py)
    x2, y2 = min(w, x2 + px), min(h, y2 + py)
    if x2 <= x1 or y2 <= y1:
        return None
    crop = cv2.resize(frame[y1:y2, x1:x2], (size, size))
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


def process_video(video_path, out_dir, video_id, cfg):
    """Extract frames, detect faces, write JPEG crops. Returns stats."""
    frames = extract_frames(video_path, cfg["frames_per_video"])
    saved = failed = 0
    for i, frame in enumerate(frames):
        face = detect_and_crop_face(frame, cfg["face_size"], cfg["face_conf_thresh"])
        if face is None:
            failed += 1
            continue
        cv2.imwrite(os.path.join(out_dir, f"{video_id}_f{i:02d}.jpg"),
                    cv2.cvtColor(face, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved += 1
    return {"video_id": video_id, "faces_saved": saved,
            "faces_failed": failed, "success": saved >= 5}

print("Preprocessing functions ready")

Face detector loaded (OpenCV ResNet-SSD, CPU backend)
Preprocessing functions ready


In [96]:
# CELL 6 - Build source-disjoint FF++ manifest and Celeb-DF manifest
#
# FF++ fake filenames are TARGET_SOURCE.mp4 (e.g. 479_706.mp4). Each source ID
# pairs with exactly one other, giving 500 isolated two-node components in the
# identity graph. Assigning whole components to a partition yields a split with
# zero source-identity leakage while retaining all 5000 videos.
# -----------------------------------------------------------------------------
def build_ff_manifest_source_disjoint(ff_root, real_folders, fake_folders, seed=42):
    rng = random.Random(seed)

    fake_records = []
    for folder in fake_folders:
        fp = os.path.join(ff_root, folder)
        for fname in sorted(os.listdir(fp)):
            if not fname.endswith(".mp4"):
                continue
            parts = fname[:-4].split("_")
            if len(parts) < 2:
                raise ValueError(f"Unexpected FF++ filename: {fname}")
            src_a, src_b = parts[:2]
            fake_records.append({
                "path"      : os.path.join(fp, fname),
                "label"     : 1,
                "label_str" : "fake",
                "category"  : folder,
                "video_id"  : f"{folder}_{fname[:-4]}",
                "src_a"     : src_a,
                "src_b"     : src_b,
            })

    pairs = sorted(set(tuple(sorted((r["src_a"], r["src_b"]))) for r in fake_records))
    flat  = [s for pair in pairs for s in pair]
    assert len(flat) == len(set(flat)), \
        "A source ID appears in more than one pair; component split unsafe."

    print("FF++ SOURCE-DISJOINT SPLIT")
    print(f"  Fake videos     : {len(fake_records):,}")
    print(f"  Source pairs    : {len(pairs):,}")
    print(f"  Unique sources  : {len(set(flat)):,}")

    rng.shuffle(pairs)
    n       = len(pairs)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)
    train_pairs = set(pairs[:n_train])
    val_pairs   = set(pairs[n_train:n_train + n_val])
    test_pairs  = set(pairs[n_train + n_val:])

    assert not (train_pairs & val_pairs)
    assert not (train_pairs & test_pairs)
    assert not (val_pairs & test_pairs)
    assert train_pairs | val_pairs | test_pairs == set(pairs)

    print(f"\n  Train pairs     : {len(train_pairs)}")
    print(f"  Val pairs       : {len(val_pairs)}")
    print(f"  Test pairs      : {len(test_pairs)}")

    def which_split(a, b):
        key = tuple(sorted((a, b)))
        if key in train_pairs: return "train"
        if key in val_pairs:   return "val"
        if key in test_pairs:  return "test"
        raise ValueError(f"Pair {key} unassigned")

    for r in fake_records:
        r["split"] = which_split(r["src_a"], r["src_b"])

    source_split = {}
    for group, split in [(train_pairs, "train"), (val_pairs, "val"), (test_pairs, "test")]:
        for pair in group:
            for src in pair:
                assert src not in source_split, f"Source {src} in multiple components"
                source_split[src] = split

    real_records = []
    for folder in real_folders:
        fp = os.path.join(ff_root, folder)
        for fname in sorted(os.listdir(fp)):
            if not fname.endswith(".mp4"):
                continue
            src = fname[:-4]
            if src not in source_split:
                raise ValueError(f"Real source {src} not in pair graph")
            real_records.append({
                "path"      : os.path.join(fp, fname),
                "label"     : 0,
                "label_str" : "real",
                "category"  : folder,
                "video_id"  : f"real_{src}",
                "src_a"     : src,
                "src_b"     : src,
                "split"     : source_split[src],
            })

    df = pd.DataFrame(fake_records + real_records)

    print("\n  FINAL MANIFEST")
    for (sp, lb), cnt in df.groupby(["split", "label_str"]).size().items():
        print(f"    {sp:<6} {lb:<5}: {cnt:,}")
    print(f"    Total       : {len(df):,}")

    tr = set(df[df.split == "train"]["src_a"]) | set(df[df.split == "train"]["src_b"])
    va = set(df[df.split == "val"]["src_a"])   | set(df[df.split == "val"]["src_b"])
    te = set(df[df.split == "test"]["src_a"])  | set(df[df.split == "test"]["src_b"])

    print("\n  LEAKAGE VERIFICATION")
    print(f"    Train/test overlap : {len(tr & te)}")
    print(f"    Train/val overlap  : {len(tr & va)}")
    print(f"    Val/test overlap   : {len(va & te)}")
    assert not (tr & te), "Source leakage between train and test"
    assert not (tr & va), "Source leakage between train and val"
    assert not (va & te), "Source leakage between val and test"
    assert df["video_id"].is_unique, "Duplicate video_id in manifest"
    print("    Zero source leakage confirmed. All video IDs unique.")

    print("\n  MANIPULATION DISTRIBUTION")
    print(df.groupby(["split", "category"]).size().to_string())
    return df


def build_celeb_manifest(celeb_root):
    records = []
    for folder, label, label_str in [("Celeb-real", 0, "real"),
                                      ("YouTube-real", 0, "real"),
                                      ("Celeb-synthesis", 1, "fake")]:
        fp = os.path.join(celeb_root, folder)
        for fname in sorted(os.listdir(fp)):
            if fname.endswith(".mp4"):
                records.append({"path": os.path.join(fp, fname), "label": label,
                                 "label_str": label_str, "category": folder,
                                 "video_id": f"{folder}_{fname[:-4]}"})
    df = pd.DataFrame(records)
    print("\nCELEB-DF MANIFEST")
    for (lb, cat), cnt in df.groupby(["label_str", "category"]).size().items():
        print(f"  {lb:<5} {cat:<18}: {cnt:,}")
    print(f"  Total: {len(df):,}")
    return df


for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        os.makedirs(f"{FF_PROCESSED}/{split}/{label}", exist_ok=True)
for label in ["real", "fake"]:
    os.makedirs(f"{CELEB_PROCESSED}/{label}", exist_ok=True)

ff_manifest    = build_ff_manifest_source_disjoint(
    FF_ROOT, FF_REAL_FOLDERS, FF_FAKE_FOLDERS, CFG["seed"])
celeb_manifest = build_celeb_manifest(CELEB_ROOT)

ff_manifest.to_csv(MANIFEST_FF, index=False)
celeb_manifest.to_csv(MANIFEST_CELEB, index=False)
print(f"\nSaved {MANIFEST_FF}")
print(f"Saved {MANIFEST_CELEB}")

FF++ SOURCE-DISJOINT SPLIT
  Fake videos     : 4,000
  Source pairs    : 500
  Unique sources  : 1,000

  Train pairs     : 350
  Val pairs       : 75
  Test pairs      : 75

  FINAL MANIFEST
    test   fake : 600
    test   real : 150
    train  fake : 2,800
    train  real : 700
    val    fake : 600
    val    real : 150
    Total       : 5,000

  LEAKAGE VERIFICATION
    Train/test overlap : 0
    Train/val overlap  : 0
    Val/test overlap   : 0
    Zero source leakage confirmed. All video IDs unique.

  MANIPULATION DISTRIBUTION
split  category      
test   Deepfakes         150
       Face2Face         150
       FaceSwap          150
       NeuralTextures    150
       original          150
train  Deepfakes         700
       Face2Face         700
       FaceSwap          700
       NeuralTextures    700
       original          700
val    Deepfakes         150
       Face2Face         150
       FaceSwap          150
       NeuralTextures    150
       original          150

C

In [10]:
# CELL 7 - Process FF++ videos into face crops(runs ~2–3 hrs)

def process_dataset(manifest, out_base, dataset_name, cfg, has_split=True):
    results = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc=dataset_name):
        out_dir = (os.path.join(out_base, row["split"], row["label_str"])
                   if has_split else os.path.join(out_base, row["label_str"]))
        os.makedirs(out_dir, exist_ok=True)
        existing = [f for f in os.listdir(out_dir) if f.startswith(row["video_id"])]
        if len(existing) >= 5:
            results.append({**row, "faces_saved": len(existing), "success": True})
            continue
        stats = process_video(row["path"], out_dir, row["label_str"], row["video_id"], cfg)
        results.append({**row, **stats})
    df_out = pd.DataFrame(results)
    print(f"\n{dataset_name} complete - "
          f"success rate: {df_out['success'].mean()*100:.1f}%  "
          f"avg faces/vid: {df_out['faces_saved'].mean():.1f}")
    return df_out

print("Processing FF++ (5000 videos - ~2–3 hrs)...")
ff_results = process_dataset(ff_manifest, f"{OUT_ROOT}/ff", "FaceForensics++", CFG, has_split=True)
ff_results.to_csv(f"{OUT_ROOT}/ff_results.csv", index=False)
print("FF++ done")

Processing FF++ (5000 videos - ~2–3 hrs)...


FaceForensics++:   0%|          | 0/5000 [00:00<?, ?it/s]


FaceForensics++ complete - success rate: 97.1%  avg faces/vid: 18.7
FF++ done


In [11]:
# CELL 8 - Process Celeb-DF (runs ~2–3 hrs)

print("Processing Celeb-DF (6529 videos - ~2–3 hrs)...")
celeb_results = process_dataset(celeb_manifest, f"{OUT_ROOT}/celeb", "Celeb-DF v2", CFG, has_split=False)
celeb_results.to_csv(f"{OUT_ROOT}/celeb_results_proc.csv", index=False)
print("Celeb-DF done")

import shutil
disk_used = shutil.disk_usage("/kaggle/working").used
print(f"\n  Disk used: {disk_used/1e9:.2f} GB / 20 GB")
print("\nPreprocessing summary:")
print(f"  FF++   : {ff_results.faces_saved.sum():,} face crops")
print(f"  Celeb  : {celeb_results.faces_saved.sum():,} face crops")

Processing Celeb-DF (6529 videos - ~2–3 hrs)...


Celeb-DF v2:   0%|          | 0/6529 [00:00<?, ?it/s]


Celeb-DF v2 complete - success rate: 96.8%  avg faces/vid: 18.2
Celeb-DF done

  Disk used: 3.09 GB / 20 GB

Preprocessing summary:
  FF++   : 93,413 face crops
  Celeb  : 119,049 face crops


In [97]:
# CELL 9 - Rebuild crop folders to match the source-disjoint split
#
# ONLY to run this if crops on disk were written under the OLD random split.
# If CELL 7 ran fresh with the v2 manifest, skip this cell entirely.
#
# This builds a clean tree at processed/ff_rebuilt, verifies it, then replaces
# processed/ff. It never leaves a crop in two places, which a copy-in-place
# reorganisation would do and which would put identical crops in train and test.
# -----------------------------------------------------------------------------

def rebuild_crop_tree(manifest_path, ff_dir):
    manifest = pd.read_csv(manifest_path)
    id_split = dict(zip(manifest["video_id"], manifest["split"]))
    id_label = dict(zip(manifest["video_id"], manifest["label_str"]))

    staging = ff_dir + "_rebuilt"
    if os.path.exists(staging):
        shutil.rmtree(staging)
    for split in ["train", "val", "test"]:
        for label in ["real", "fake"]:
            os.makedirs(os.path.join(staging, split, label), exist_ok=True)

    # index every crop currently on disk, wherever it sits
    found, orphaned, seen_ids = 0, 0, set()
    for split in ["train", "val", "test"]:
        for label in ["real", "fake"]:
            src_dir = os.path.join(ff_dir, split, label)
            if not os.path.exists(src_dir):
                continue
            for fname in os.listdir(src_dir):
                if not fname.endswith(".jpg"):
                    continue
                video_id = fname.rsplit("_f", 1)[0]
                if video_id not in id_split:
                    orphaned += 1
                    continue
                dst = os.path.join(staging, id_split[video_id],
                                    id_label[video_id], fname)
                shutil.copy2(os.path.join(src_dir, fname), dst)
                seen_ids.add(video_id)
                found += 1

    print("REBUILD SUMMARY")
    print(f"  Crops relocated       : {found:,}")
    print(f"  Orphaned crops        : {orphaned:,} (video_id not in manifest)")
    print(f"  Distinct videos found : {len(seen_ids):,} of {len(manifest):,}")

    print("\n  New tree counts")
    total_new = 0
    for split in ["train", "val", "test"]:
        for label in ["real", "fake"]:
            d = os.path.join(staging, split, label)
            n = len([f for f in os.listdir(d) if f.endswith(".jpg")])
            total_new += n
            print(f"    {split:<6} {label:<5}: {n:>8,}")
    print(f"    {'TOTAL':<12}: {total_new:>8,}")

    # verify no crop landed in more than one split
    per_split_ids = {}
    for split in ["train", "val", "test"]:
        ids = set()
        for label in ["real", "fake"]:
            d = os.path.join(staging, split, label)
            for f in os.listdir(d):
                if f.endswith(".jpg"):
                    ids.add(f.rsplit("_f", 1)[0])
        per_split_ids[split] = ids

    tv = per_split_ids["train"] & per_split_ids["test"]
    tval = per_split_ids["train"] & per_split_ids["val"]
    vt = per_split_ids["val"] & per_split_ids["test"]
    print(f"\n  Video ID overlap train/test : {len(tv)}")
    print(f"  Video ID overlap train/val  : {len(tval)}")
    print(f"  Video ID overlap val/test   : {len(vt)}")
    assert not tv and not tval and not vt, "Video appears in multiple splits after rebuild"
    assert total_new == found, "Crop count mismatch"
    print("\n  Verification passed.")
    print(f"  Staging tree ready at: {staging}")
    print("  Review the numbers above, then run the swap cell below.")
    return staging

STAGING = rebuild_crop_tree(MANIFEST_FF, FF_PROCESSED)

REBUILD SUMMARY
  Crops relocated       : 93,179
  Orphaned crops        : 0 (video_id not in manifest)
  Distinct videos found : 4,854 of 5,000

  New tree counts
    train  real :   13,080
    train  fake :   52,292
    val    real :    2,722
    val    fake :   10,954
    test   real :    2,819
    test   fake :   11,312
    TOTAL       :   93,179

  Video ID overlap train/test : 0
  Video ID overlap train/val  : 0
  Video ID overlap val/test   : 0

  Verification passed.
  Staging tree ready at: /kaggle/working/processed/ff_rebuilt
  Review the numbers above, then run the swap cell below.


In [78]:
# CELL 9B - Swap in the rebuilt tree and delete the old one
# Run only after CELL 9 verification passes and the counts look right.

old_backup = FF_PROCESSED + "_old"
if os.path.exists(old_backup):
    shutil.rmtree(old_backup)
os.rename(FF_PROCESSED, old_backup)
os.rename(STAGING, FF_PROCESSED)
shutil.rmtree(old_backup)

print("Swap complete. Final crop counts:")
total = 0
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        d = os.path.join(FF_PROCESSED, split, label)
        n = len([f for f in os.listdir(d) if f.endswith(".jpg")])
        total += n
        print(f"  {split:<6} {label:<5}: {n:>8,}")
print(f"  {'TOTAL':<12}: {total:>8,}")
print(f"\nDisk used: {shutil.disk_usage('/kaggle/working').used/1e9:.2f} GB")

Swap complete. Final crop counts:
  train  real :   13,080
  train  fake :   52,292
  val    real :    2,722
  val    fake :   10,954
  test   real :    2,819
  test   fake :   11,312
  TOTAL       :   93,179

Disk used: 3.32 GB


In [ ]:
# ============================================================
# ONE-TIME REPAIR LOG (already applied - do not re-run)
# Documents how 54 unrecoverable videos were identified and
# removed from ff_manifest_v2.csv. Kept for reproducibility
# and for the dissertation methodology section.
# ============================================================
# check what the manifest actually contains right now
manifest = pd.read_csv(MANIFEST_FF)
print(f"Total rows        : {len(manifest)}")
print(f"\nBreakdown:")
print(manifest.groupby(["split", "label_str"]).size())
print(f"\nPer-category (should be 700/150/150 if this is the correct build):")
print(manifest.groupby(["split", "category"]).size())
manifest = pd.read_csv(MANIFEST_FF)
expected_ids = set(manifest["video_id"])

found_ids = set()
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        d = os.path.join(FF_PROCESSED, split, label)
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.endswith(".jpg"):
                    found_ids.add(f.rsplit("_f", 1)[0])

missing_ids = expected_ids - found_ids
print(f"Videos in manifest    : {len(expected_ids)}")
print(f"Videos with crops     : {len(found_ids)}")
print(f"Videos missing crops  : {len(missing_ids)}")
if missing_ids:
    print(list(missing_ids)[:10])

In [103]:
# ============================================================
# ONE-TIME REPAIR LOG (already applied - do not re-run)
# Documents how 54 unrecoverable videos were identified and
# removed from ff_manifest_v2.csv. Kept for reproducibility
# and for the dissertation methodology section.
# ============================================================
missing_rows = manifest[manifest["video_id"].isin(missing_ids)]
print(f"Reprocessing {len(missing_rows)} videos...")

reprocess_log = []
for _, row in tqdm(missing_rows.iterrows(), total=len(missing_rows)):
    out_dir = os.path.join(FF_PROCESSED, row["split"], row["label_str"])
    os.makedirs(out_dir, exist_ok=True)
    stats = process_video(row["path"], out_dir, row["video_id"], CFG)
    reprocess_log.append(stats)

df_reprocess = pd.DataFrame(reprocess_log)
print(f"\nSuccess rate on retry : {df_reprocess['success'].mean()*100:.1f}%")
print(f"Still zero faces      : {(df_reprocess['faces_saved']==0).sum()}")

# re-verify against the manifest
found_ids_after = set()
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        d = os.path.join(FF_PROCESSED, split, label)
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.endswith(".jpg"):
                    found_ids_after.add(f.rsplit("_f", 1)[0])

still_missing = expected_ids - found_ids_after
print(f"\nVideos still missing crops: {len(still_missing)}")
if still_missing:
    print(list(still_missing))

Reprocessing 145 videos...


  0%|          | 0/145 [00:00<?, ?it/s]


Success rate on retry : 0.0%
Still zero faces      : 54

Videos still missing crops: 54
['Deepfakes_686_696', 'NeuralTextures_444_655', 'Deepfakes_374_407', 'Face2Face_444_655', 'real_856', 'FaceSwap_856_881', 'NeuralTextures_877_886', 'NeuralTextures_199_181', 'Deepfakes_976_954', 'Deepfakes_990_008', 'NeuralTextures_580_524', 'real_880', 'FaceSwap_880_135', 'Deepfakes_681_711', 'FaceSwap_580_524', 'Deepfakes_877_886', 'Deepfakes_856_881', 'Deepfakes_580_524', 'real_990', 'real_348', 'Face2Face_856_881', 'Face2Face_348_202', 'Face2Face_502_504', 'FaceSwap_444_655', 'FaceSwap_877_886', 'Deepfakes_880_135', 'Face2Face_874_291', 'FaceSwap_990_008', 'Face2Face_282_238', 'real_199', 'NeuralTextures_227_169', 'Deepfakes_444_655', 'real_282', 'real_227', 'Face2Face_880_135', 'NeuralTextures_174_964', 'FaceSwap_374_407', 'NeuralTextures_502_504', 'Deepfakes_227_169', 'NeuralTextures_282_238', 'NeuralTextures_348_202', 'NeuralTextures_880_135', 'FaceSwap_282_238', 'Deepfakes_348_202', 'Neural

In [109]:
# ============================================================
# ONE-TIME REPAIR LOG (already applied - do not re-run)
# Documents how 54 unrecoverable videos were identified and
# removed from ff_manifest_v2.csv. Kept for reproducibility
# and for the dissertation methodology section.
# ============================================================
# confirm the identity-clustering pattern
import re

failed_sources = set()
for vid in still_missing:
    match = re.search(r'(\d{3})(?:_(\d{3}))?$', vid)
    if match:
        failed_sources.add(match.group(1))
        if match.group(2):
            failed_sources.add(match.group(2))

print(f"Unique source identities behind the 54 failures: {len(failed_sources)}")
print(sorted(failed_sources))
# remove unrecoverable videos from the manifest
manifest = pd.read_csv(MANIFEST_FF)
before = len(manifest)
manifest_clean = manifest[~manifest["video_id"].isin(still_missing)].copy()
after = len(manifest_clean)

print(f"Manifest rows before : {before}")
print(f"Manifest rows after  : {after}")
print(f"Dropped              : {before - after}")

print("\nNew split breakdown:")
print(manifest_clean.groupby(["split", "label_str"]).size())

print("\nNew per-category breakdown:")
print(manifest_clean.groupby(["split", "category"]).size())

manifest_clean.to_csv(MANIFEST_FF, index=False)
print(f"\nSaved cleaned manifest to {MANIFEST_FF}")
# final verification: every manifest row now has crops on disk
manifest = pd.read_csv(MANIFEST_FF)
expected_ids = set(manifest["video_id"])

found_ids = set()
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        d = os.path.join(FF_PROCESSED, split, label)
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.endswith(".jpg"):
                    found_ids.add(f.rsplit("_f", 1)[0])

missing_final = expected_ids - found_ids
orphan_crops  = found_ids - expected_ids

print(f"Manifest videos       : {len(expected_ids)}")
print(f"Videos with crops     : {len(found_ids & expected_ids)}")
print(f"Still missing         : {len(missing_final)}")
print(f"Orphan crops on disk  : {len(orphan_crops)} (crops for videos no longer in manifest)")

Unique source identities behind the 54 failures: 36
['008', '135', '169', '174', '181', '199', '202', '227', '238', '282', '291', '348', '374', '407', '444', '502', '504', '524', '556', '580', '588', '655', '681', '686', '696', '711', '856', '874', '877', '880', '881', '886', '954', '964', '976', '990']
Manifest rows before : 4946
Manifest rows after  : 4946
Dropped              : 0

New split breakdown:
split  label_str
test   fake          597
       real          149
train  fake         2772
       real          694
val    fake          586
       real          148
dtype: int64

New per-category breakdown:
split  category      
test   Deepfakes         149
       Face2Face         149
       FaceSwap          150
       NeuralTextures    149
       original          149
train  Deepfakes         692
       Face2Face         694
       FaceSwap          693
       NeuralTextures    693
       original          694
val    Deepfakes         146
       Face2Face         147
       FaceSw

In [110]:
manifest = pd.read_csv(MANIFEST_FF)
print(f"Final manifest: {len(manifest)} videos")
print(manifest.groupby(["split", "label_str"]).size())

Final manifest: 4946 videos
split  label_str
test   fake          597
       real          149
train  fake         2772
       real          694
val    fake          586
       real          148
dtype: int64


In [15]:
#=============================================================================
# RESUME CELL A - run after any session restart, before Section 4
# Rebuilds imports, paths, config. Assumes preprocessing is already done.
# Run it instead of re-running Cells 1–9B after a restart.
#=============================================================================

import os, cv2, json, random, time, io, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from tqdm.notebook import tqdm
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix, roc_curve)
import timm
import matplotlib.pyplot as plt
from PIL import Image as PILImage
warnings.filterwarnings('ignore')

# Paths
FF_ROOT         = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
CELEB_ROOT      = "/kaggle/input/datasets/reubensuju/celeb-df-v2"
OUT_ROOT        = "/kaggle/working/processed"
FF_PROCESSED    = "/kaggle/working/processed/ff"
CELEB_PROCESSED = "/kaggle/working/processed/celeb"
CKPT_DIR        = "/kaggle/working/checkpoints"
MODEL_DIR       = "/kaggle/working/models"
GRADCAM_DIR     = "/kaggle/working/gradcam"
MC_DIR          = "/kaggle/working/mc_dropout"
EXP_DIR         = "/kaggle/working/experiments"
MANIFEST_FF     = f"{OUT_ROOT}/ff_manifest_v2.csv"
MANIFEST_CELEB  = f"{OUT_ROOT}/celeb_manifest.csv"

for d in [CKPT_DIR, MODEL_DIR, GRADCAM_DIR, MC_DIR, EXP_DIR]:
    os.makedirs(d, exist_ok=True)

CFG = {"frames_per_video": 20, "face_size": 224,
       "face_conf_thresh": 0.90, "batch_size": 32, "seed": 42}
FF_REAL_FOLDERS = ["original"]
FF_FAKE_FOLDERS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Manifest present: {os.path.exists(MANIFEST_FF)}")
print("Run CELL 10 next.")

Device: cpu - CPU


# SECTION 4- MODEL DEFINITIONS

In [3]:
# CELL 10 - Transforms and dataset classes

# Both branches receive the SAME augmented image. The spatial branch gets it
# ImageNet-normalised; the frequency branch gets raw [0,1] pixels, because FFT
# analysis is meaningful on the original pixel distribution rather than on a
# mean-shifted, variance-scaled tensor.

class GeneralisationAugment:
    """Simulates cross-dataset variation: JPEG, blur, sharpness, sensor noise."""
    def __call__(self, img):
        from PIL import ImageFilter, ImageEnhance
        if random.random() < 0.3:
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=random.randint(50, 95))
            buf.seek(0)
            img = PILImage.open(buf).convert('RGB')
        if random.random() < 0.2:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.1, 1.5)))
        if random.random() < 0.2:
            img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.5))
        if random.random() < 0.2:
            arr = np.array(img).astype(np.float32)
            arr = np.clip(arr + np.random.normal(0, random.uniform(2, 8), arr.shape),
                           0, 255).astype(np.uint8)
            img = PILImage.fromarray(arr)
        return img

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

# augmentation only, returns a PIL image
train_augment = transforms.Compose([
    GeneralisationAugment(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.08),
    transforms.RandomGrayscale(p=0.05),
])

norm_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])
raw_transform = transforms.Compose([
    transforms.ToTensor(),
])
# kept for single-image inference elsewhere in the notebook
val_transform = norm_transform


class FaceDataset(Dataset):
    """Face crops from root/split/label/*.jpg"""
    def __init__(self, root, split, augment=None):
        self.samples, self.augment = [], augment
        for label_str, label in [("real", 0), ("fake", 1)]:
            folder = os.path.join(root, split, label_str)
            if not os.path.exists(folder):
                continue
            for fname in os.listdir(folder):
                if fname.endswith(".jpg"):
                    self.samples.append((os.path.join(folder, fname), label))
        random.shuffle(self.samples)
        n_real = sum(1 for _, l in self.samples if l == 0)
        n_fake = len(self.samples) - n_real
        print(f"  {split:<6}: {len(self.samples):>8,} crops "
              f"(real={n_real:,}, fake={n_fake:,})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        pil = PILImage.fromarray(img)
        if self.augment is not None:
            pil = self.augment(pil)          # one augmentation, shared
        return norm_transform(pil), raw_transform(pil), \
               torch.tensor(label, dtype=torch.float32)


class CelebDataset(Dataset):
    """Face crops from root/label/*.jpg, evaluation only, no augmentation."""
    def __init__(self, root):
        self.samples = []
        for label_str, label in [("real", 0), ("fake", 1)]:
            folder = os.path.join(root, label_str)
            if not os.path.exists(folder):
                continue
            for fname in os.listdir(folder):
                if fname.endswith(".jpg"):
                    self.samples.append((os.path.join(folder, fname), label))
        n_real = sum(1 for _, l in self.samples if l == 0)
        n_fake = len(self.samples) - n_real
        print(f"  Celeb : {len(self.samples):>8,} crops "
              f"(real={n_real:,}, fake={n_fake:,})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        pil = PILImage.fromarray(img)
        return norm_transform(pil), raw_transform(pil), \
               torch.tensor(label, dtype=torch.float32)


def make_weighted_sampler(dataset):
    """Balances real/fake exposure per epoch. FF++ is 4:1 fake:real."""
    labels  = [label for _, label in dataset.samples]
    counts  = [labels.count(0), labels.count(1)]
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(weights, len(weights), replacement=True)

print("Transforms and dataset classes ready")

Transforms and dataset classes ready


In [4]:
# CELL 11 - DataLoaders

print("Loading datasets")
train_ds = FaceDataset(FF_PROCESSED, "train", augment=train_augment)
val_ds   = FaceDataset(FF_PROCESSED, "val",   augment=None)
test_ds  = FaceDataset(FF_PROCESSED, "test",  augment=None)
celeb_ds = CelebDataset(CELEB_PROCESSED)

BS = CFG["batch_size"]
train_loader = DataLoader(train_ds, batch_size=BS,
                           sampler=make_weighted_sampler(train_ds),
                           num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS*2, shuffle=False,
                           num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS*2, shuffle=False,
                           num_workers=2, pin_memory=True)
celeb_loader = DataLoader(celeb_ds, batch_size=BS*2, shuffle=False,
                           num_workers=2, pin_memory=True)

print(f"\nBatches - train {len(train_loader)}, val {len(val_loader)}, "
      f"test {len(test_loader)}, celeb {len(celeb_loader)}")


Loading datasets...
  train   :   65577 images(real=13181, fake=52396)
  val     :   14109 images(real=2772, fake=11337)
  test    :   13727 images(real=2719, fake=11008)
  Celeb-DF: 119,049 images (real=15671, fake=103378)

DataLoaders ready
   Train batches : 2050
   Val batches   : 221
   Test batches  : 215
   Celeb batches : 1861


In [5]:
# CELL 12 - Model architectures

class SpatialBranch(nn.Module):
    """EfficientNet-B4 backbone. Detects blending and texture artefacts."""
    def __init__(self, pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b4", pretrained=pretrained,
                                           num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features            # 1792
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat_dim, 512),
            nn.ReLU(), nn.Dropout(dropout / 2), nn.Linear(512, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(1)

    def get_features(self, x):
        return self.backbone(x)


class FrequencyBranchV2(nn.Module):
    """
    2D FFT magnitude spectrum through a small CNN. Takes raw [0,1] pixels so
    the spectrum reflects the true pixel distribution. Retains 2D structure
    rather than collapsing to a radial profile, so directional GAN artefacts
    remain visible to the convolutions.
    """
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x_raw):
        gray    = 0.299 * x_raw[:, 0] + 0.587 * x_raw[:, 1] + 0.114 * x_raw[:, 2]
        mag     = torch.abs(torch.fft.fftshift(torch.fft.fft2(gray)))
        log_mag = torch.log1p(mag)
        b  = log_mag.size(0)
        mn = log_mag.view(b, -1).min(1).values.view(b, 1, 1)
        mx = log_mag.view(b, -1).max(1).values.view(b, 1, 1)
        log_mag = (log_mag - mn) / (mx - mn + 1e-8)
        log_mag = F.interpolate(log_mag.unsqueeze(1), size=(112, 112),
                                 mode='bilinear', align_corners=False)
        return self.cnn(log_mag).squeeze(-1).squeeze(-1)


class DualBranchFinal(nn.Module):
    """
    Attention-gated fusion. The gate outputs alpha in [0,1] per sample and the
    fused representation is s + alpha * P(f). Alpha is therefore directly
    interpretable as how much frequency evidence the model chose to use.
    """
    def __init__(self, spatial_ckpt=None, dropout=0.4):
        super().__init__()
        self.spatial = SpatialBranch(pretrained=True, dropout=dropout)
        if spatial_ckpt and os.path.exists(spatial_ckpt):
            ckpt = torch.load(spatial_ckpt, map_location="cpu", weights_only=False)
            self.spatial.load_state_dict(ckpt["model_state"])
            print(f"  Spatial weights loaded (val AUC={ckpt['val_auc']:.4f})")
        self.freq      = FrequencyBranchV2()
        self.freq_proj = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 1792))
        self.gate = nn.Sequential(
            nn.Linear(1792 + 256, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid())
        self.classifier = nn.Sequential(
            nn.Linear(1792, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(512, 1))

    def forward(self, x, x_raw):
        s     = self.spatial.get_features(x)
        f     = self.freq(x_raw)
        alpha = self.gate(torch.cat([s, f], dim=1))
        blend = s + alpha * self.freq_proj(f)
        return self.classifier(blend).squeeze(1), alpha

    def forward_pred(self, x, x_raw):
        logits, _ = self.forward(x, x_raw)
        return logits


_s  = SpatialBranch(pretrained=False).to(device)
_f  = FrequencyBranchV2().to(device)
_x  = torch.randn(2, 3, 224, 224).to(device)
_xr = torch.rand(2, 3, 224, 224).to(device)
print(f"SpatialBranch feature dim     : {list(_s.get_features(_x).shape)}")
print(f"FrequencyBranchV2 feature dim : {list(_f(_xr).shape)}")
del _s, _f, _x, _xr

SpatialBranch features: [2, 1792]
FrequencyBranchV2 features: [2, 256]


# SECTION 5: TRAINING: SPATIAL BRANCH (BASELINE)

In [6]:
# CELL 14 - Training utilities

def train_one_epoch(model, loader, optimizer, criterion, device, dual=False):
    model.train()

    total_loss, correct, total = 0.0, 0, 0
    for imgs, imgs_raw, labels in loader:
        imgs, imgs_raw, labels = imgs.to(device), imgs_raw.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model.forward_pred(imgs, imgs_raw) if dual else model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += ((torch.sigmoid(logits) > 0.5).float() == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device, dual=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels, all_alphas = [], [], []
    with torch.no_grad():
        for imgs, imgs_raw, labels in loader:
            imgs, imgs_raw, labels = imgs.to(device), imgs_raw.to(device), labels.to(device)
            if dual:
                logits, alpha = model.forward(imgs, imgs_raw)
                all_alphas.extend(alpha.cpu().numpy().flatten())
            else:
                logits = model(imgs)
            loss  = criterion(logits, labels)
            probs = torch.sigmoid(logits)
            total_loss += loss.item() * imgs.size(0)
            correct    += ((probs > 0.5).float() == labels).sum().item()
            total      += imgs.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    auc       = roc_auc_score(all_labels, all_probs)
    avg_alpha = float(np.mean(all_alphas)) if all_alphas else None
    return total_loss / total, correct / total, auc, avg_alpha


def full_metrics(model, loader, device, dual=False):
    from sklearn.metrics import roc_curve
    model.eval()
    all_probs, all_labels = [], []
    t_start   = time.time()
    n_samples = 0
    with torch.no_grad():
        for imgs, imgs_raw, labels in tqdm(loader, desc="Evaluating"):
            imgs, imgs_raw = imgs.to(device), imgs_raw.to(device)
            if dual:
                logits, _ = model.forward(imgs, imgs_raw)
            else:
                logits = model(imgs)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(labels.numpy())
            n_samples += imgs.size(0)
    elapsed   = time.time() - t_start
    all_preds = [1 if p > 0.5 else 0 for p in all_probs]
    cm        = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    fpr_val   = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr_val   = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    fpr_curve, tpr_curve, _ = roc_curve(all_labels, all_probs)
    fnr_curve = 1 - tpr_curve
    eer_idx   = np.argmin(np.abs(fpr_curve - fnr_curve))
    eer       = float((fpr_curve[eer_idx] + fnr_curve[eer_idx]) / 2)
    return {
        "auc"               : roc_auc_score(all_labels, all_probs),
        "acc"               : accuracy_score(all_labels, all_preds),
        "prec"              : precision_score(all_labels, all_preds),
        "rec"               : recall_score(all_labels, all_preds),
        "f1"                : f1_score(all_labels, all_preds),
        "fpr"               : float(fpr_val),
        "fnr"               : float(fnr_val),
        "eer"               : eer,
        "cm"                : cm.tolist(),
        "time_per_sample_ms": (elapsed / n_samples) * 1000,
    }
def video_level_metrics(model, manifest_path, split, processed_root, device, dual=False):
    from sklearn.metrics import roc_curve
    manifest  = pd.read_csv(manifest_path)
    subset    = manifest[manifest["split"] == split]
    model.eval()

    all_vid_probs, all_vid_labels = [], []
    t_start = time.time()

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f"Video-level ({split})"):
        label_str = row["label_str"]
        folder    = os.path.join(processed_root, split, label_str)
        crops     = sorted([f for f in os.listdir(folder)
                             if f.startswith(row["video_id"]) and f.endswith(".jpg")])
        if not crops:
            continue

        frame_probs = []
        for crop_fname in crops:
            img     = cv2.cvtColor(cv2.imread(os.path.join(folder, crop_fname)), cv2.COLOR_BGR2RGB)
            pil_img = PILImage.fromarray(img)
            tensor  = val_transform(pil_img).unsqueeze(0).to(device)
            raw_t   = raw_transform(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                if dual:
                    logit, _ = model.forward(tensor, raw_t)
                else:
                    logit = model(tensor)
                frame_probs.append(torch.sigmoid(logit).item())

        all_vid_probs.append(float(np.mean(frame_probs)))
        all_vid_labels.append(int(row["label"]))

    elapsed   = time.time() - t_start
    all_preds = [1 if p > 0.5 else 0 for p in all_vid_probs]
    cm        = confusion_matrix(all_vid_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    fpr_val   = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr_val   = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    fpr_curve, tpr_curve, _ = roc_curve(all_vid_labels, all_vid_probs)
    fnr_curve = 1 - tpr_curve
    eer_idx   = np.argmin(np.abs(fpr_curve - fnr_curve))
    eer       = float((fpr_curve[eer_idx] + fnr_curve[eer_idx]) / 2)

    n_videos = len(all_vid_probs)
    return {
        "auc"                  : roc_auc_score(all_vid_labels, all_vid_probs),
        "acc"                  : accuracy_score(all_vid_labels, all_preds),
        "prec"                 : precision_score(all_vid_labels, all_preds),
        "rec"                  : recall_score(all_vid_labels, all_preds),
        "f1"                   : f1_score(all_vid_labels, all_preds),
        "fpr"                  : float(fpr_val),
        "fnr"                  : float(fnr_val),
        "eer"                  : eer,
        "cm"                   : cm.tolist(),
        "n_videos"             : n_videos,
        "time_per_video_ms"    : (elapsed / n_videos) * 1000,
    }
print("Training utilities ready")

Training utilities ready


In [ ]:
# CELL 15 - Train spatial branch (EfficientNet-B4 baseline)
# ~10 epochs × ~17 mins = ~170 mins. Best checkpoint saved automatically.

spatial_model = SpatialBranch(pretrained=True, dropout=0.4).to(device)
criterion     = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.5]).to(device))
optimizer_sp  = optim.AdamW(spatial_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_sp  = optim.lr_scheduler.CosineAnnealingLR(optimizer_sp, T_max=10, eta_min=1e-6)
N_EPOCHS_SP   = 10

print(f"Training SpatialBranch - {N_EPOCHS_SP} epochs\n")
best_sp_auc, sp_history = 0.0, []

for epoch in range(1, N_EPOCHS_SP + 1):
    t0 = time.time()
    tr_loss, tr_acc  = train_one_epoch(spatial_model, train_loader, optimizer_sp, criterion, device)
    vl_loss, vl_acc, vl_auc, _ = evaluate(spatial_model, val_loader, criterion, device)
    scheduler_sp.step()
    elapsed = time.time() - t0
    print(f"Epoch {epoch:02d}/{N_EPOCHS_SP} | "
          f"train loss={tr_loss:.4f} acc={tr_acc:.3f} | "
          f"val loss={vl_loss:.4f} acc={vl_acc:.3f} auc={vl_auc:.4f} | "
          f"lr={optimizer_sp.param_groups[0]['lr']:.2e} | {elapsed:.0f}s")
    sp_history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": vl_loss, "val_acc": vl_acc, "val_auc": vl_auc})
    if vl_auc > best_sp_auc:
        best_sp_auc = vl_auc
        torch.save({"epoch": epoch, "model_state": spatial_model.state_dict(),
                    "optimizer": optimizer_sp.state_dict(), "val_auc": vl_auc},
                   f"{CKPT_DIR}/spatial_best.pth")
        print(f"           ↑ best spatial model saved (AUC={vl_auc:.4f})")

with open(f"{CKPT_DIR}/spatial_history.json", "w") as f:
    json.dump(sp_history, f, indent=2)
print(f"\nSpatial training done - best val AUC: {best_sp_auc:.4f}")
print(f"   Target ≥0.85 → {'PASSED' if best_sp_auc >= 0.85 else 'not yet'}")

In [ ]:
# CELL 16 - Evaluate spatial branch on FF++ test set

ckpt = torch.load(f"{CKPT_DIR}/spatial_best.pth", map_location=device, weights_only=False)
spatial_model.load_state_dict(ckpt["model_state"])
print(f"Loaded best spatial model (epoch {ckpt['epoch']}, val AUC={ckpt['val_auc']:.4f})\n")

sp_metrics = full_metrics(spatial_model, test_loader, device, dual=False)
cm         = np.array(sp_metrics["cm"])

print(f"  SPATIAL BRANCH - FF++ TEST SET")
print(f"  AUC       : {sp_metrics['auc']:.4f}  (target ≥0.85)")
print(f"  Accuracy  : {sp_metrics['acc']:.4f}")
print(f"  Precision : {sp_metrics['prec']:.4f}")
print(f"  Recall    : {sp_metrics['rec']:.4f}")
print(f"  F1        : {sp_metrics['f1']:.4f}")
print(f"  Confusion matrix:")
print(f"              Pred Real  Pred Fake")
print(f"  True Real :   {cm[0][0]:>6}     {cm[0][1]:>6}")
print(f"  True Fake :   {cm[1][0]:>6}     {cm[1][1]:>6}")

print(f"\n  {'TARGET MET' if sp_metrics['auc'] >= 0.85 else 'TARGET MISSED'}")

print(f"  FPR        : {sp_metrics['fpr']:.4f}  (false alarm rate on real videos)")
print(f"  FNR        : {sp_metrics['fnr']:.4f}  (miss rate on fake videos)")
print(f"  EER        : {sp_metrics['eer']:.4f}")
print(f"  Time/sample: {sp_metrics['time_per_sample_ms']:.1f} ms")

with open(f"{CKPT_DIR}/spatial_test_results.json", "w") as f:
    json.dump({k: float(v) if not isinstance(v, list) else v
               for k, v in sp_metrics.items()}, f, indent=2)
print(f"Saved → spatial_test_results.json")

SPATIAL_TEST_AUC = sp_metrics["auc"]  # store for comparison later

# SECTION 6: TRAINING: FINAL DUAL-BRANCH MODEL

In [ ]:
# CELL 17 - Retrain dual model with spatial FULLY FROZEN throughout
# Runtime: ~6 epochs × ~18 mins = ~108 mins

# Fresh dual model - load spatial weights
dual_model = DualBranchFinal(
    spatial_ckpt = f"{CKPT_DIR}/spatial_best.pth",
    dropout      = 0.4,
).to(device)

# Freeze spatial branch completely - never unfreeze
for param in dual_model.spatial.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in dual_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in dual_model.parameters())
print(f"DualBranchFinal - spatial FROZEN")
print(f"   Total params     : {total/1e6:.1f}M")
print(f"   Trainable params : {trainable/1e6:.1f}M  (freq + fusion only)")

criterion     = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.5]).to(device))
best_dual_auc = 0.0
dual_history  = []

# Single training stage - freq + fusion only, higher LR since no fine-tuning risk
optimizer_dual = optim.AdamW([
    {"params": dual_model.freq.parameters(),       "lr": 5e-4},
    {"params": dual_model.freq_proj.parameters(),  "lr": 5e-4},
    {"params": dual_model.gate.parameters(),       "lr": 5e-4},
    {"params": dual_model.classifier.parameters(), "lr": 5e-4},
], weight_decay=1e-4)

N_EPOCHS_DUAL = 8
scheduler_dual = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_dual, T_max=N_EPOCHS_DUAL, eta_min=1e-6
)

print(f"\nTraining freq + fusion (spatial frozen) - {N_EPOCHS_DUAL} epochs\n")

for epoch in range(1, N_EPOCHS_DUAL + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(
        dual_model, train_loader, optimizer_dual, criterion, device, dual=True
    )
    vl_loss, vl_acc, vl_auc, avg_alpha = evaluate(
        dual_model, val_loader, criterion, device, dual=True
    )
    scheduler_dual.step()
    elapsed = time.time() - t0

    print(f"Epoch {epoch:02d}/{N_EPOCHS_DUAL} | "
          f"train loss={tr_loss:.4f} acc={tr_acc:.3f} | "
          f"val auc={vl_auc:.4f} alpha={avg_alpha:.3f} | "
          f"lr={optimizer_dual.param_groups[0]['lr']:.2e} | {elapsed:.0f}s")

    dual_history.append({"epoch": epoch, "val_auc": vl_auc,
                          "train_loss": tr_loss, "avg_alpha": avg_alpha})

    if vl_auc > best_dual_auc:
        best_dual_auc = vl_auc
        torch.save({
            "epoch"      : epoch,
            "model_state": dual_model.state_dict(),
            "val_auc"    : vl_auc,
            "avg_alpha"  : avg_alpha,
        }, f"{CKPT_DIR}/dual_best.pth")
        print(f"           ↑ best dual saved (AUC={vl_auc:.4f}, alpha={avg_alpha:.3f})")

with open(f"{CKPT_DIR}/dual_history.json", "w") as f:
    json.dump(dual_history, f, indent=2)

print(f"\nDual training done")
print(f"   Best val AUC   : {best_dual_auc:.4f}")
print(f"   Spatial val AUC: {best_sp_auc:.4f}")
print(f"   Watch: val_auc > {best_sp_auc:.4f} needed on TEST SET for Obj 3")

In [ ]:
# ************************************************************************************ #
# RESUME CELL B - Run after session restart (after training is complete)
#  Rebuilds all variables AND loads both trained models from checkpoints.
# run resume Cell A and then run Cell 13 

SPATIAL_TEST_AUC = None  # will be set after evaluation

# Reload spatial model
spatial_model = SpatialBranch(pretrained=False).to(device)
ckpt_sp = torch.load(f"{CKPT_DIR}/spatial_best.pth", map_location=device, weights_only=False)
spatial_model.load_state_dict(ckpt_sp["model_state"])
print(f"Spatial model loaded - val AUC={ckpt_sp['val_auc']:.4f}")

# Reload dual model
dual_model = DualBranchFinal(spatial_ckpt=None, dropout=0.4).to(device)
ckpt_dual  = torch.load(f"{CKPT_DIR}/dual_best.pth", map_location=device, weights_only=False)
dual_model.load_state_dict(ckpt_dual["model_state"])
print(f"Dual model loaded - val AUC={ckpt_dual['val_auc']:.4f}")

# Load previous spatial test result if available
if os.path.exists(f"{CKPT_DIR}/spatial_test_results.json"):
    with open(f"{CKPT_DIR}/spatial_test_results.json") as f:
        _prev = json.load(f)
    SPATIAL_TEST_AUC = _prev["auc"]
    print(f"Previous spatial test AUC loaded: {SPATIAL_TEST_AUC:.4f}")

print("Session fully restored - ready to run from Section 7 onward")

# SECTION 7: EVALUATION & ABLATION

In [ ]:
# CELL 18 - Evaluate dual model on FF++ test set

ckpt_dual = torch.load(f"{CKPT_DIR}/dual_best.pth", map_location=device, weights_only=False)
dual_model.load_state_dict(ckpt_dual["model_state"])
print(f"Loaded best dual model (epoch {ckpt_dual['epoch']}, val AUC={ckpt_dual['val_auc']:.4f})\n")

dual_metrics = full_metrics(dual_model, test_loader, device, dual=True)
cm_dual      = np.array(dual_metrics["cm"])

print(f"  DUAL-BRANCH - FF++ TEST SET")
print(f"  AUC       : {dual_metrics['auc']:.4f}")
print(f"  Accuracy  : {dual_metrics['acc']:.4f}")
print(f"  Precision : {dual_metrics['prec']:.4f}")
print(f"  Recall    : {dual_metrics['rec']:.4f}")
print(f"  F1        : {dual_metrics['f1']:.4f}")
print(f"  Confusion matrix:")
print(f"              Pred Real  Pred Fake")
print(f"  True Real :   {cm_dual[0][0]:>6}     {cm_dual[0][1]:>6}")
print(f"  True Fake :   {cm_dual[1][0]:>6}     {cm_dual[1][1]:>6}")
print(f"  FPR        : {dual_metrics['fpr']:.4f}  (false alarm rate on real videos)")
print(f"  FNR        : {dual_metrics['fnr']:.4f}  (miss rate on fake videos)")
print(f"  EER        : {dual_metrics['eer']:.4f}")
print(f"  Time/sample: {dual_metrics['time_per_sample_ms']:.1f} ms")

DUAL_TEST_AUC    = dual_metrics["auc"]

if SPATIAL_TEST_AUC is None:
    if os.path.exists(f"{CKPT_DIR}/spatial_test_results.json"):
        with open(f"{CKPT_DIR}/spatial_test_results.json") as f:
            SPATIAL_TEST_AUC = json.load(f)["auc"]
    else:
        raise RuntimeError("Run CELL 16 first to get spatial test AUC")

with open(f"{CKPT_DIR}/dual_test_results.json", "w") as f:
    json.dump({k: float(v) if not isinstance(v, list) else v
               for k, v in dual_metrics.items()}, f, indent=2)
print(f"\nSaved → dual_test_results.json")

In [ ]:
# CELL 19 - Cross-dataset evaluation on Celeb-DF (zero retraining)

celeb_metrics = full_metrics(dual_model, celeb_loader, device, dual=True)
gen_gap       = DUAL_TEST_AUC - celeb_metrics["auc"]
cm_celeb      = np.array(celeb_metrics["cm"])

print(f"{'='*52}")
print(f"  CROSS-DATASET: CELEB-DF v2")
print(f"  (trained on FF++ only - zero retraining)")
print(f"{'='*52}")
print(f"  AUC             : {celeb_metrics['auc']:.4f}  (target ≥0.75)")
print(f"  Accuracy        : {celeb_metrics['acc']:.4f}")
print(f"  F1              : {celeb_metrics['f1']:.4f}")
print(f"  Confusion matrix:")
print(f"              Pred Real  Pred Fake")
print(f"  True Real :   {cm_celeb[0][0]:>6}     {cm_celeb[0][1]:>6}")
print(f"  True Fake :   {cm_celeb[1][0]:>6}     {cm_celeb[1][1]:>6}")
print(f"\n  FF++ test AUC   : {DUAL_TEST_AUC:.4f}")
print(f"  Celeb-DF AUC    : {celeb_metrics['auc']:.4f}")
print(f"  Gen. gap        : {gen_gap:.4f}")
print(f"\n  AUC target      : {'MET' if celeb_metrics['auc'] >= 0.75 else 'MISSED'}")

with open(f"{CKPT_DIR}/celeb_results.json", "w") as f:
    json.dump({"celeb_auc": float(celeb_metrics["auc"]),
               "celeb_acc": float(celeb_metrics["acc"]),
               "ff_test_auc": float(DUAL_TEST_AUC),
               "gen_gap": float(gen_gap),
               "auc_target_met": bool(celeb_metrics["auc"] >= 0.75)
              }, f, indent=2)
print(f"Saved → celeb_results.json")
CELEB_AUC = celeb_metrics["auc"]

In [ ]:
# CELL 20 - Ablation study + per-manipulation breakdown

print("Ablation summary:")
print(f"  Spatial-only (EfficientNet-B4) : {SPATIAL_TEST_AUC:.4f}")
print(f"  Dual-branch (Spatial + FreqV2) : {DUAL_TEST_AUC:.4f}")
print(f"  Improvement                    : {DUAL_TEST_AUC - SPATIAL_TEST_AUC:+.4f}")
print(f"  Objective 3 met                : "
      f"{'YES' if DUAL_TEST_AUC > SPATIAL_TEST_AUC else 'NO'}")

print("Per-manipulation breakdown (each type vs real videos):")
ff_manifest   = pd.read_csv(f"{OUT_ROOT}/ff_manifest.csv")
test_manifest = ff_manifest[ff_manifest["split"] == "test"]

# get all real video probabilities once
real_rows   = test_manifest[test_manifest["category"] == "original"]
real_probs, real_labels = [], []
for _, row in real_rows.iterrows():
    folder = os.path.join(FF_PROCESSED, "test", "real")
    crops  = sorted([f for f in os.listdir(folder)
                     if f.startswith(row["video_id"]) and f.endswith(".jpg")])
    if not crops:
        continue
    frame_probs = []
    for crop_fname in crops:
        img     = cv2.cvtColor(cv2.imread(os.path.join(folder, crop_fname)), cv2.COLOR_BGR2RGB)
        pil_img = PILImage.fromarray(img)
        tensor  = val_transform(pil_img).unsqueeze(0).to(device)
        raw_t   = raw_transform(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            logit, _ = dual_model.forward(tensor, raw_t)
            frame_probs.append(torch.sigmoid(logit).item())
    real_probs.append(float(np.mean(frame_probs)))
    real_labels.append(0)

print(f"\n  {'Category':<20} {'AUC':>7}  {'Acc':>7}  {'FPR':>7}  {'FNR':>7}  {'N':>5}")
print("  " + "-" * 60)

per_manip_results = {}
for cat in ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]:
    fake_rows = test_manifest[test_manifest["category"] == cat]
    fake_probs, fake_labels = [], []
    for _, row in fake_rows.iterrows():
        folder = os.path.join(FF_PROCESSED, "test", "fake")
        crops  = sorted([f for f in os.listdir(folder)
                         if f.startswith(row["video_id"]) and f.endswith(".jpg")])
        if not crops:
            continue
        frame_probs = []
        for crop_fname in crops:
            img     = cv2.cvtColor(cv2.imread(os.path.join(folder, crop_fname)), cv2.COLOR_BGR2RGB)
            pil_img = PILImage.fromarray(img)
            tensor  = val_transform(pil_img).unsqueeze(0).to(device)
            raw_t   = raw_transform(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                logit, _ = dual_model.forward(tensor, raw_t)
                frame_probs.append(torch.sigmoid(logit).item())
        fake_probs.append(float(np.mean(frame_probs)))
        fake_labels.append(1)

    all_probs  = real_probs  + fake_probs
    all_labels = real_labels + fake_labels
    all_preds  = [1 if p > 0.5 else 0 for p in all_probs]
    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, all_preds)
    cm  = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    print(f"  {cat:<20} {auc:>7.4f}  {acc:>7.4f}  {fpr:>7.4f}  {fnr:>7.4f}  {len(fake_rows):>5}")
    per_manip_results[cat] = {"auc": float(auc), "acc": float(acc),
                               "fpr": float(fpr), "fnr": float(fnr), "n": len(fake_rows)}

with open(f"{CKPT_DIR}/per_manipulation_results.json", "w") as f:
    json.dump(per_manip_results, f, indent=2)
print("\nPer-manipulation results saved")

with open(f"{CKPT_DIR}/ablation_summary.json", "w") as f:
    json.dump({"spatial_only": float(SPATIAL_TEST_AUC),
               "dual_branch":  float(DUAL_TEST_AUC),
               "improvement":  float(DUAL_TEST_AUC - SPATIAL_TEST_AUC),
               "celeb_auc":    float(CELEB_AUC),
               "gen_gap":      float(gen_gap)}, f, indent=2)
print("\n Ablation results saved")

In [ ]:
# resume cell C: run after evaluation is complete, before explainability
# requires: Resume Cell A + Cell 13 + Resume Cell B already run

# reload dual model
ckpt_dual = torch.load(f"{CKPT_DIR}/dual_best.pth", map_location=device, weights_only=False)
dual_model.load_state_dict(ckpt_dual["model_state"])
dual_model.eval()

# reload saved results
with open(f"{CKPT_DIR}/spatial_test_results.json") as f:
    sp_metrics = json.load(f)
with open(f"{CKPT_DIR}/dual_test_results.json") as f:
    dual_metrics = json.load(f)
with open(f"{CKPT_DIR}/celeb_results.json") as f:
    _celeb = json.load(f)

SPATIAL_TEST_AUC = sp_metrics["auc"]
DUAL_TEST_AUC    = dual_metrics["auc"]
CELEB_AUC        = _celeb["celeb_auc"]
gen_gap          = _celeb["gen_gap"]

# reinitialise grad-cam (needs dual_model loaded above)
#gradcam = GradCAM(dual_model)
# GradCAM class is defined in CELL 21 — NEED TO run that cell before the explainability cells

print(f"Resumed - spatial AUC={SPATIAL_TEST_AUC:.4f}, dual AUC={DUAL_TEST_AUC:.4f}")
print(f"Celeb-DF AUC={CELEB_AUC:.4f}, gap={gen_gap:.4f}")
print("Ready to run from explainability cells onward")

# SECTION 8: EXPLAINABILITY

In [ ]:
# CELL 21 - Grad-CAM implementation

class GradCAM:
    """Grad-CAM for EfficientNet-B4 - hooks last conv block."""
    def __init__(self, model):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target = model.spatial.backbone.blocks[-1]
        target.register_forward_hook(lambda m,i,o: setattr(self, 'activations', o.detach()))
        target.register_full_backward_hook(lambda m,gi,go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, img_tensor, img_raw_tensor):
        self.model.eval()
        img_tensor = img_tensor.requires_grad_(True)
        logits, _  = self.model.forward(img_tensor, img_raw_tensor)
        prob       = torch.sigmoid(logits).item()
        self.model.zero_grad()
        logits.backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam     = torch.relu((weights * self.activations).sum(dim=1)).squeeze(0).cpu().numpy()
        cam     = cam - cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        cam = cv2.resize(cam, (224, 224))
        return cam, prob


def overlay_heatmap(face_rgb, heatmap, alpha=0.45):
    colormap = cv2.cvtColor(
        cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB)
    return (alpha * colormap + (1 - alpha) * face_rgb).astype(np.uint8)

gradcam = GradCAM(dual_model)
print("Grad-CAM ready (target: dual_model.spatial.backbone.blocks[-1])")

In [ ]:
# CELL 22 - Generate Grad-CAM for 100 test samples (75 fake + 25 real)

os.makedirs(f"{GRADCAM_DIR}/fake", exist_ok=True)
os.makedirs(f"{GRADCAM_DIR}/real", exist_ok=True)

test_samples  = FaceDataset(FF_PROCESSED, "test", val_transform)
fake_samples  = [(p,l) for p,l in test_samples.samples if l == 1][:75]
real_samples  = [(p,l) for p,l in test_samples.samples if l == 0][:25]
selected      = fake_samples + real_samples
random.shuffle(selected)

print(f"Generating Grad-CAM for {len(selected)} samples...")
gc_records = []
dual_model.eval()

for i, (img_path, true_label) in enumerate(tqdm(selected, desc="Grad-CAM")):
    raw_rgb    = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img_tensor = val_transform(PILImage.fromarray(raw_rgb)).unsqueeze(0).to(device)
    raw_tensor    = raw_transform(PILImage.fromarray(raw_rgb)).unsqueeze(0).to(device)
    heatmap, prob = gradcam.generate(img_tensor, raw_tensor)
    pred_label = 1 if prob > 0.5 else 0
    correct    = (pred_label == true_label)
    overlay    = overlay_heatmap(raw_rgb, heatmap)
    composite  = np.hstack([raw_rgb, overlay])
    label_str  = "fake" if true_label == 1 else "real"
    pred_str   = "fake" if pred_label == 1 else "real"
    fname      = f"{i:03d}_true{label_str}_pred{pred_str}_{'ok' if correct else 'wrong'}.jpg"
    cv2.imwrite(os.path.join(GRADCAM_DIR, label_str, fname),
                cv2.cvtColor(composite, cv2.COLOR_RGB2BGR))
    gc_records.append({"idx": i, "path": img_path, "true_label": true_label,
                        "pred_label": pred_label, "prob": prob, "correct": correct})

df_gc      = pd.DataFrame(gc_records)
gc_acc     = df_gc["correct"].mean()
print(f"\nGrad-CAM complete - {len(df_gc)} samples, accuracy: {gc_acc:.3f}")
for lbl, name in [(1,"fake"),(0,"real")]:
    sub = df_gc[df_gc.true_label == lbl]
    print(f"   {name} accuracy: {sub['correct'].mean():.3f} ({sub['correct'].sum()}/{len(sub)})")
df_gc.to_csv(f"{GRADCAM_DIR}/gradcam_results.csv", index=False)

In [ ]:
# resume cell D: run if session dies between grad-cam and mc dropout ---
# requires: Resume Cell A + Cell 13 + Resume Cell B + Resume Cell C already run

import pandas as pd
df_gc = pd.read_csv(f"{GRADCAM_DIR}/gradcam_results.csv")
print(f"Grad-CAM results reloaded: {len(df_gc)} samples")
print(f"Accuracy: {df_gc['correct'].mean():.3f}")

In [ ]:
# CELL 23 - Visualise Grad-CAM grid

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle("Grad-CAM Explainability - Deepfake Detection\nLeft=Original  Right=Heatmap",
             fontsize=13, fontweight='bold')
categories = [
    (df_gc[(df_gc.true_label==1)&(df_gc.correct==True)],  "Fake → Correctly detected"),
    (df_gc[(df_gc.true_label==1)&(df_gc.correct==False)], "Fake → Missed"),
    (df_gc[(df_gc.true_label==0)&(df_gc.correct==True)],  "Real → Correctly passed"),
    (df_gc[(df_gc.true_label==0)&(df_gc.correct==False)], "Real → False alarm"),
]
for row_idx, (subset, title) in enumerate(categories):
    axes[row_idx, 0].set_ylabel(title, fontsize=9)
    for col_offset, (_, row) in enumerate(subset.head(3).iterrows()):
        label_str = "fake" if row.true_label == 1 else "real"
        pred_str  = "fake" if row.pred_label == 1 else "real"
        fname     = f"{int(row.idx):03d}_true{label_str}_pred{pred_str}_{'ok' if row.correct else 'wrong'}.jpg"
        fpath     = os.path.join(GRADCAM_DIR, label_str, fname)
        if os.path.exists(fpath):
            comp  = cv2.cvtColor(cv2.imread(fpath), cv2.COLOR_BGR2RGB)
            for ci, img_part in enumerate([comp[:,:224,:], comp[:,224:,:]]):
                ax = axes[row_idx, col_offset*2 + ci]
                ax.imshow(img_part)
                if ci == 0:
                    ax.set_title(f"p={row.prob:.2f}", fontsize=8)
                ax.axis("off")
for r in range(4):
    for c in range(6):
        if not axes[r,c].images:
            axes[r,c].axis("off")
plt.tight_layout()
plt.savefig(f"{GRADCAM_DIR}/gradcam_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grad-CAM grid saved → gradcam_grid.png")

In [ ]:
# CELL 24 - MC Dropout uncertainty estimation (T=50, 100 samples)

os.makedirs(MC_DIR, exist_ok=True)

def mc_dropout_predict(model, img_tensor, img_raw_tensor, n_passes=50):
    def enable_dropout(m):
        if isinstance(m, nn.Dropout):
            m.train()
    model.eval()
    model.apply(enable_dropout)
    probs = []
    with torch.no_grad():
        for _ in range(n_passes):
            logit, _ = model.forward(img_tensor, img_raw_tensor)
            probs.append(torch.sigmoid(logit).item())
    probs = np.array(probs)
    return {"mean_prob"  : float(probs.mean()),
            "variance"   : float(probs.var()),
            "std"        : float(probs.std()),
            "prediction" : "fake" if probs.mean() > 0.5 else "real"}

mc_records = []
for _, row in tqdm(df_gc.iterrows(), total=len(df_gc), desc="MC Dropout"):
    raw_rgb    = cv2.cvtColor(cv2.imread(row.path), cv2.COLOR_BGR2RGB)
    pil_img    = PILImage.fromarray(raw_rgb)
    img_tensor = val_transform(pil_img).unsqueeze(0).to(device)
    raw_tensor = raw_transform(pil_img).unsqueeze(0).to(device)
    mc_out     = mc_dropout_predict(dual_model, img_tensor, raw_tensor, n_passes=50)
    mc_records.append({"path": row.path, "true_label": int(row.true_label),
                        "correct": bool(row.correct), **mc_out})

df_mc = pd.DataFrame(mc_records)
df_mc.to_csv(f"{MC_DIR}/mc_results.csv", index=False)
print(f"\nMC Dropout complete - {len(df_mc)} samples")
print(f"   Mean predictive variance : {df_mc['variance'].mean():.5f}")
print(f"   Mean predictive std      : {df_mc['std'].mean():.5f}")
print(f"   Mean variance            : {df_mc['variance'].mean():.5f}")
print(f"   Correct preds var        : {df_mc[df_mc.correct==True]['variance'].mean():.5f}")
print(f"   Wrong preds var          : {df_mc[df_mc.correct==False]['variance'].mean():.5f}")

In [ ]:
# CELL 25 - Plot MC Dropout uncertainty

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("MC Dropout Uncertainty Analysis (T=50 passes)", fontsize=12, fontweight='bold')
# Plot 1: variance by correctness
axes[0].hist(df_mc[df_mc.correct==True]['variance'],  bins=15, alpha=0.7, label='Correct',   color='steelblue')
axes[0].hist(df_mc[df_mc.correct==False]['variance'], bins=15, alpha=0.7, label='Incorrect', color='tomato')
axes[0].set_xlabel('Prediction variance'); axes[0].set_ylabel('Count')
axes[0].set_title('Variance: correct vs incorrect'); axes[0].legend()
# Plot 2: mean_prob vs variance
axes[1].scatter(df_mc['mean_prob'], df_mc['variance'],
                c=df_mc['true_label'], cmap='RdYlGn', alpha=0.6, s=40)
axes[1].set_xlabel('Mean probability'); axes[1].set_ylabel('Variance')
axes[1].set_title('Variance vs probability\n(green=fake, red=real)')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
# Plot 3: variance by true label
axes[2].boxplot([df_mc[df_mc.true_label==0]['variance'],
                 df_mc[df_mc.true_label==1]['variance']], labels=['Real','Fake'])
axes[2].set_ylabel('Variance'); axes[2].set_title('Variance by true label')
plt.tight_layout()
plt.savefig(f"{MC_DIR}/mc_uncertainty.png", dpi=150, bbox_inches='tight')
plt.show()
print("Uncertainty plot saved → mc_uncertainty.png")

In [ ]:
# Cell 26 - Final results summary

print("DEEPFAKE DETECTION - RESULTS SUMMARY")
print("Group 9, University of the West of England")
print()

print("Objective 1 - FF++ AUC >= 0.85")
print(f"  Model        : DualBranchFinal (EfficientNet-B4 + FrequencyBranchV2)")
print(f"  Test AUC     : {DUAL_TEST_AUC:.4f}")
print(f"  Accuracy     : {dual_metrics['acc']:.4f}")
print(f"  F1           : {dual_metrics['f1']:.4f}")
print(f"  Result       : {'PASSED' if DUAL_TEST_AUC >= 0.85 else 'MISSED'}")
print()

print("Objective 2 - Celeb-DF AUC >= 0.75 (cross-dataset evaluation)")
print(f"  Test AUC     : {CELEB_AUC:.4f}")
print(f"  Gen. gap     : {gen_gap:.4f}  (FF++ to Celeb-DF drop)")
print(f"  Note         : generalisation gap quantified for dissertation analysis")
print(f"  Result       : {'PASSED' if CELEB_AUC >= 0.75 else 'MISSED'}")
print()

print("Objective 3 - Dual-branch AUC > Spatial-only AUC")
print(f"  Spatial-only : {SPATIAL_TEST_AUC:.4f}")
print(f"  Dual-branch  : {DUAL_TEST_AUC:.4f}")
print(f"  Difference   : {DUAL_TEST_AUC - SPATIAL_TEST_AUC:+.4f}")
print(f"  Avg alpha    : {ckpt_dual.get('avg_alpha', 'N/A')}")
print(f"  Note         : alpha near 0 means the gate learned to suppress")
print(f"                 frequency features; spatial branch dominates")
print(f"  Result       : {'PASSED' if DUAL_TEST_AUC > SPATIAL_TEST_AUC else 'MARGINAL'}")
print()

print("Objective 4 - Explainability")
print("  Grad-CAM     : 100 samples generated")
print("  MC Dropout   : T=50, 100 samples")
print("  Note         : model shows very low variance (overconfident)")
print()

print("Objective 5 - Web application")
print(" Status : Flask + React app built and running")
print()

print("Key output files:")
files = {
    "spatial_best.pth":              f"{CKPT_DIR}/spatial_best.pth",
    "dual_best.pth":                 f"{CKPT_DIR}/dual_best.pth",
    "spatial_test_results.json":     f"{CKPT_DIR}/spatial_test_results.json",
    "dual_test_results.json":        f"{CKPT_DIR}/dual_test_results.json",
    "celeb_results.json":            f"{CKPT_DIR}/celeb_results.json",
    "ablation_summary.json":         f"{CKPT_DIR}/ablation_summary.json",
    "per_manipulation_results.json": f"{CKPT_DIR}/per_manipulation_results.json",
    "gradcam_grid.png":              f"{GRADCAM_DIR}/gradcam_grid.png",
    "mc_uncertainty.png":            f"{MC_DIR}/mc_uncertainty.png",
}
for name, path in files.items():
    found = "found" if os.path.exists(path) else "missing"
    print(f"  {found:<8} {name}")

In [1]:
import os

def inspect_working(root="/kaggle/working"):
    total_bytes = 0
    total_files = 0

    for path, dirs, files in os.walk(root):
        total_files += len(files)
        for name in files:
            try:
                total_bytes += os.path.getsize(os.path.join(path, name))
            except OSError:
                pass

    print(f"Files: {total_files:,}")
    print(f"Size:  {total_bytes / 1024**3:.2f} GB")

inspect_working()

Files: 212,484
Size:  2.92 GB


In [3]:
# identity leakage check - run this before doing anything else
# FF++ fake video filenames follow the pattern e.g. "Deepfakes_479_706"
# where 479 and 706 are source identity indices from the original dataset

ff_manifest = pd.read_csv(f"{OUT_ROOT}/ff_manifest.csv")
fake_rows   = ff_manifest[ff_manifest["label"] == 1].copy()

# extract the two source identity numbers from each fake video_id
# e.g. "Deepfakes_479_706" → src_a=479, src_b=706
fake_rows["src_a"] = fake_rows["video_id"].str.extract(r'_(\d+)_\d+$')
fake_rows["src_b"] = fake_rows["video_id"].str.extract(r'_\d+_(\d+)$')

train_fake = fake_rows[fake_rows["split"] == "train"]
test_fake  = fake_rows[fake_rows["split"] == "test"]

train_ids  = set(train_fake["src_a"].dropna()) | set(train_fake["src_b"].dropna())
test_ids   = set(test_fake["src_a"].dropna())  | set(test_fake["src_b"].dropna())

overlap    = train_ids & test_ids

print(f"Unique source identity indices in train : {len(train_ids)}")
print(f"Unique source identity indices in test  : {len(test_ids)}")
print(f"Overlapping identity indices            : {len(overlap)}")
print(f"Overlap rate (test identities in train) : {len(overlap)/len(test_ids)*100:.1f}%")
print()

# also check real videos - these are named "real_123" so no identity pairing
real_rows  = ff_manifest[ff_manifest["label"] == 0].copy()
real_train = set(real_rows[real_rows["split"]=="train"]["video_id"])
real_test  = set(real_rows[real_rows["split"]=="test"]["video_id"])
real_overlap = real_train & real_test
print(f"Real video overlap (should be 0)        : {len(real_overlap)}")

if len(overlap) == 0:
    print("\nNo source identity overlap found - split is clean")
elif len(overlap) / len(test_ids) < 0.20:
    print("\nModerate overlap - document as limitation, proceed with current split")
else:
    print("\nHigh overlap - consider rebuilding with source-aware split")

Unique source identity indices in train : 1000
Unique source identity indices in test  : 738
Overlapping identity indices            : 738
Overlap rate (test identities in train) : 100.0%

Real video overlap (should be 0)        : 0

High overlap - consider rebuilding with source-aware split


In [4]:
# examine source identity structure before rebuilding
ff_manifest = pd.read_csv(f"{OUT_ROOT}/ff_manifest.csv")
fake_rows   = ff_manifest[ff_manifest["label"] == 1].copy()
fake_rows["src_a"] = fake_rows["video_id"].str.extract(r'_(\d+)_\d+$')
fake_rows["src_b"] = fake_rows["video_id"].str.extract(r'_\d+_(\d+)$')

# find all unique source identities
all_src = set(fake_rows["src_a"].dropna()) | set(fake_rows["src_b"].dropna())
print(f"Total unique source identities : {len(all_src)}")

# how many fake videos per source identity on average
src_counts = pd.concat([
    fake_rows["src_a"].dropna(),
    fake_rows["src_b"].dropna()
]).value_counts()
print(f"Avg fake videos per identity   : {src_counts.mean():.1f}")
print(f"Min / Max                      : {src_counts.min()} / {src_counts.max()}")

# real videos
real_rows = ff_manifest[ff_manifest["label"] == 0].copy()
print(f"\nReal videos total              : {len(real_rows)}")
print(f"Unique real video IDs          : {real_rows['video_id'].nunique()}")

Total unique source identities : 1000
Avg fake videos per identity   : 8.0
Min / Max                      : 8 / 8

Real videos total              : 1000
Unique real video IDs          : 1000
